In [14]:
!pip install datasets
!pip install jiwer
!pip install pandas

In [15]:
import torch
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer
import re
from torch.utils.data import DataLoader
import pandas as pd

In [3]:
def string_preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def get_edit_distance(org_sentence, pred_sentence):
    org_words, pred_words = org_sentence.split(), pred_sentence.split()
    len_org, len_pred = len(org_words), len(pred_words)
    dp = dict()

    for index in range(len_pred + 1):
        dp[(index, 0)] = index
    for index in range(len_org + 1):
        dp[(0, index)] = index

    for index1 in range(1, len_pred + 1):
        for index2 in range(1, len_org + 1):
            if pred_words[index1 - 1] == org_words[index2 - 1]:
                dp[(index1, index2)] = dp[(index1 - 1, index2 - 1)]
            else:
                dp[(index1, index2)] = 1 + min(
                    dp[(index1 - 1, index2)],
                    dp[(index1, index2 - 1)],
                    dp[(index1 - 1, index2 - 1)]
                )

    return dp[(len_pred, len_org)]


def compute_word_error_rate(original_texts, predicted_texts):
    total_words = 0
    total_edit_distances = 0
    for index in range(len(original_texts)):
        org_sentence = string_preprocess(original_texts[index])
        pred_sentence = string_preprocess(predicted_texts[index])
        edit_distance = get_edit_distance(org_sentence, pred_sentence)
        total_edit_distances += edit_distance
        total_words += len(org_sentence.split())

    return float(total_edit_distances) / total_words

In [4]:
MODEL_NAME    = "openai/whisper-base"
SPLIT         = "validation"
TARGET_SR     = 16_000
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
DEVICE

device(type='cuda')

In [6]:
ds_small = (
        load_dataset("edinburghcstr/edacc", split=SPLIT)
        .cast_column("audio", Audio(sampling_rate=TARGET_SR))
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/6.95k [00:00<?, ?B/s]

(…)-00000-of-00006-17ec18f4d1c3d587.parquet:   0%|          | 0.00/457M [00:00<?, ?B/s]

(…)-00001-of-00006-6f3978e4f6163671.parquet:   0%|          | 0.00/478M [00:00<?, ?B/s]

(…)-00002-of-00006-5a1d86079a8c9228.parquet:   0%|          | 0.00/446M [00:00<?, ?B/s]

...

(…)-00007-of-00010-ea5ed4464ecff3c9.parquet:   0%|          | 0.00/439M [00:00<?, ?B/s]

(…)-00008-of-00010-d1aa19b51ad423da.parquet:   0%|          | 0.00/409M [00:00<?, ?B/s]

(…)-00009-of-00010-648ce5002d124496.parquet:   0%|          | 0.00/299M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/9848 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9289 [00:00<?, ? examples/s]

In [7]:
ds_small

Dataset({
    features: ['speaker', 'text', 'accent', 'raw_accent', 'gender', 'l1', 'audio'],
    num_rows: 9848
})

In [8]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model     = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(DEVICE).eval()

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 512)
      (layers): ModuleList(
        (0-5): 6 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=False)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          

In [11]:
import time
import torch.nn.functional as F

expected_length = 3000

BATCH_SIZE = 16
dl = DataLoader(ds_small, batch_size=BATCH_SIZE, collate_fn=lambda x: x)

references, hypotheses, accents = [], [], []
batch_count = 0
print(f"start, {time.time()}")

with torch.no_grad():
    for batch in dl:
        audio_arrays = [sample["audio"]["array"] for sample in batch]
        batch_texts  = [sample["text"] for sample in batch]
        batch_accents = [sample["accent"] for sample in batch]

        inputs      = processor(
            audio_arrays,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True,
            return_attention_mask=True,
        )
        input_feats = inputs.input_features.to(DEVICE)

        #### ADDING THIS BECAUSE IT IS GIVING AN ERROR OTHERWISE - Whisper expects the mel input features to be of length 3000, but found 2170.
        T = input_feats.shape[-1]
        if T < expected_length:
            pad_amount   = expected_length - T
            input_feats = F.pad(
                input_feats,
                (0, pad_amount),
                value=processor.feature_extractor.padding_value
            )
        #############

        pred_ids   = model.generate(input_feats)
        pred_texts = processor.batch_decode(pred_ids, skip_special_tokens=True)

        references.extend(batch_texts)
        hypotheses.extend(pred_texts)
        accents.extend(batch_accents)

        batch_count += 1
        if batch_count % 10 == 0:
            print(f"Processed {batch_count}, {time.time()}")

overall_wer = compute_word_error_rate(references, hypotheses) * 100
print(f"\nOverall WER on {len(ds_small)} examples: {overall_wer:.2f}%")

start, 1746214041.87631
Processed 10, 1746214069.195145
Processed 20, 1746214088.9132893
Processed 30, 1746214115.230006
Processed 40, 1746214143.8889787
Processed 50, 1746214184.9784992
Processed 60, 1746214207.1876886
Processed 70, 1746214232.850013
Processed 80, 1746214255.7557466
Processed 90, 1746214280.859591
Processed 100, 1746214311.2275777
Processed 110, 1746214329.5607824
Processed 120, 1746214345.6929
Processed 130, 1746214361.6808412
Processed 140, 1746214396.056624
Processed 150, 1746214435.2459636
Processed 160, 1746214452.587004
Processed 170, 1746214481.3665426
Processed 180, 1746214501.9845552
Processed 190, 1746214550.6705978
Processed 200, 1746214589.4664924
Processed 210, 1746214632.6767023
Processed 220, 1746214657.17266
Processed 230, 1746214680.63767
Processed 240, 1746214705.7690935
Processed 250, 1746214732.741779
Processed 260, 1746214755.9138668
Processed 270, 1746214786.652014
Processed 280, 1746214810.89405
Processed 290, 1746214830.431956
Processed 300, 17

In [22]:
df = pd.DataFrame({'accent': accents, 'ref_sentence': references, 'hyp_sentence': hypotheses})
unique_accents = df["accent"].unique()
unique_accents

result = {'accent': [], 'count': [], 'WER': []}

for accent in unique_accents:
    temp_df = df[df["accent"] == accent]
    acc_ref, acc_pred = list(temp_df["ref_sentence"]), list(temp_df["hyp_sentence"])
    wer = compute_word_error_rate(acc_ref, acc_pred)
    result['accent'].append(accent)
    result['count'].append(temp_df.shape[0])
    result['WER'].append(wer)

result_df = pd.DataFrame(result)
print(result_df)

                      accent  count       WER
0   Southern British English   1190  0.574178
1      Mainstream US English   1096  0.419121
2                   European    229  0.369211
3             Indian English    373  0.561698
4                    Italian    602  0.467881
5                     German     73  0.277091
6                      Dutch     98  0.547271
7                   Japanese    137  0.965108
8                    Chinese    447  0.424531
9                     French     55  0.905047
10                 Bulgarian    215  0.649660
11                   Catalan    301  0.602868
12                   Spanish    297  0.446672
13                Don't know    586  0.484312
14            Latin American     84  0.297436
15          Eastern European    862  0.408616
16        Indonesian English    324  0.966645
17                  Egyptian    469  0.362331
18     South African English    246  0.354874
19          Scottish English    248  0.439036
20          Jamaican English    29

In [ ]:
print(f"Computed WER: {overall_wer:.2f}%")
jiver_wer = wer([string_preprocess(x) for x in references], [string_preprocess(x) for x in hypotheses])
jiwer_wer_without_string_preprocess = wer(references, hypotheses)
print(f"jiwer.wer's WER: {jiver_wer * 100: .2f}%")
print(f"jiwer.wer without string preprocess: {jiwer_wer_without_string_preprocess * 100: .2f}%")

Computed WER: 48.50%
jiwer.wer's WER:  48.50%
jiwer.wer without string preprocess:  109.88%


In [ ]:
result = []

for ref, hyp in zip(references, hypotheses):
    result.append([ref, hyp])

print(result[:5])

[['C ELEVEN DASH P ONE', ' C11-P1.'], ['C ELEVEN DASH P TWO', ' c11-p2.'], ['IGNORE_TIME_SEGMENT_IN_SCORING', ' Please call Stella, ask her to bring these things with her from the store. Six spoons of fresh snow peas, five thick slabs of blue cheese, and maybe a snack for her brother Bob. We also need a small plastic snake and a big toy frog for the kids. She can scoop these things into three red bags and we will go meet her Wednesday at the train station.'], ['IGNORE_TIME_SEGMENT_IN_SCORING', ' Please call Stella, ask her to bring these things with her from the store. Six spoons of fresh snow peas, five thick slabs of blue cheese, and maybe a snack for her brother Bob. We also need a small plastic snake and a big toy frog for the kids. She can scoop these things into three red bags, and we will go meet her Wednesday at the train station.'], ['OKAY NOW FOR A REGULAR CONVERSATION SO UH WOULD YOU RATHER GO TO THE BEACH TODAY OR DO CLUB GOLF DISCUSS', ' Okay, now for a regular conversatio

In [ ]:
# testing with mozilla common voice without batching
import itertools
N_SAMPLES  = 10000
common_voice = load_dataset(
    "mozilla-foundation/common_voice_11_0", # for some reason mozilla-foundation/common_voice_13_0 and mozilla-foundation/common_voice_17_0 are not working
    "en",
    split="train",
    streaming=True
).cast_column("audio", Audio(sampling_rate=TARGET_SR))

moz_small = itertools.islice(common_voice, N_SAMPLES)

moz_references, moz_hypotheses, accents = [], [], []
index = 0
print(f"start, {time.time()}")
for sample in moz_small:
    wav = sample["audio"]["array"]
    moz_references.append(sample["sentence"])
    accents.append(sample["accent"])

    inputs = processor(wav, sampling_rate=TARGET_SR, return_tensors="pt")
    input_feats = inputs.input_features.to(DEVICE)

    pred_ids = model.generate(input_feats)

    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    moz_hypotheses.append(pred_text)

    index += 1
    if index % 100 == 0:
        print(f"{index}, {time.time()}")

overall_wer = compute_word_error_rate(moz_references, moz_hypotheses) * 100
print(f"\nOverall WER on {len(ds_small)} examples: {overall_wer:.2f}%")

start, 1746150907.1043797


Reading metadata...: 948736it [00:20, 46517.54it/s]


100, 1746150947.5201626
200, 1746150967.8543127
300, 1746150989.564885
400, 1746151010.4310145
500, 1746151030.1371524
600, 1746151053.7576191
700, 1746151077.5613573
800, 1746151098.4262388
900, 1746151119.4006743
1000, 1746151140.1572485
1100, 1746151159.7718232
1200, 1746151180.222896
1300, 1746151211.0116875
1400, 1746151234.7848842
1500, 1746151255.5132494
1600, 1746151275.8620043
1700, 1746151296.7659497
1800, 1746151323.9319897
1900, 1746151347.545353
2000, 1746151368.7024767
2100, 1746151399.1304584
2200, 1746151420.357495
2300, 1746151451.916497
2400, 1746151479.71712
2500, 1746151499.2208302
2600, 1746151524.140043
2700, 1746151544.8176975
2800, 1746151565.098594
2900, 1746151585.262708
3000, 1746151609.4775913
3100, 1746151634.4805398
3200, 1746151662.4012609
3300, 1746151682.2407992
3400, 1746151705.381338
3500, 1746151729.1355836
3600, 1746151750.1048105
3700, 1746151781.1261294
3800, 1746151809.2141964
3900, 1746151836.9059577
4000, 1746151861.3229218
4100, 1746151882.951